# Step 2 — PI-CAI Fold 0 İndirme ve Veri Keşfi

**Amaç:** Zenodo'dan PI-CAI Public Training fold 0 (~14 GB) indir, unzip et, klasör yapısını keşfet, örnek bir hastayı görselleştir.

**Strateji:**
- Zip'i **Colab local disk** (`/content/`)'e indir — Drive'a doğrudan indirme çok yavaş
- Unzip'i de local'de yap
- Sonra **sadece seçtiğimiz 20 hastayı** Drive'a kopyala (kalıcılık için)
- Tam fold gerekli değil — demo için 10-20 hasta yeter

**Önkoşul:** `step1_setup.ipynb` çalıştırıldı, kütüphaneler kurulu, Drive bağlı.

⚠️ **Colab Free session 12 saat sürer, idle 90 dk.** İndirme 30-60 dk sürebilir, ekranda kalmaya çalış.

## 2.1 — Drive bağla + path setup

Eğer step1 oturumundaysan bu cellı atlayabilirsin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Prostate_MRI_Project')
CODE_DIR     = PROJECT_ROOT / 'code'
INPUT_DIR    = PROJECT_ROOT / 'input'
IMAGES_DIR   = INPUT_DIR / 'images'           # Drive (kalıcı subset)
LABELS_DIR   = INPUT_DIR / 'picai_labels'
OUTPUT_DIR   = PROJECT_ROOT / 'output'

# Colab local disk (geçici, hızlı)
LOCAL_WORK   = Path('/content/picai_work')
LOCAL_ZIP    = LOCAL_WORK / 'fold0.zip'
LOCAL_IMAGES = LOCAL_WORK / 'images'

for d in [IMAGES_DIR, LOCAL_WORK, LOCAL_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

print('Drive root:', PROJECT_ROOT)
print('Local work:', LOCAL_WORK)

## 2.2 — Disk durumunu kontrol et

Fold 0 zip ~14 GB, unzip sonrası ~14 GB daha → ~30 GB geçici alan gerekli. Colab `/content` ~100 GB sunar.

In [ ]:
!df -h /content

## 2.3 — Zenodo'dan fold 0'ı indir

`curl -C -` ile **resume-capable** indirme. Eğer Colab kopsa, hücreyi tekrar çalıştır, kaldığı yerden devam eder.

**Süre tahmini:** Colab'ın bağlantı hızıyla 20-60 dk.

In [ ]:
import os
os.chdir(LOCAL_WORK)

ZENODO_URL = 'https://zenodo.org/api/records/6624726/files/picai_public_images_fold0.zip/content'

# -C - : resume
# -L   : redirect follow
# -o   : output filename
!curl -C - -L "{ZENODO_URL}" -o "{LOCAL_ZIP}"

In [ ]:
# İndirme boyutu kontrolü
size_gb = LOCAL_ZIP.stat().st_size / 1e9
print(f'Zip boyutu: {size_gb:.2f} GB')
assert size_gb > 10, 'Zip eksik indirilmiş olabilir, hücreyi tekrar çalıştır.'

## 2.4 — Unzip

Hızlı işlem (~5-10 dk), Colab local disk'e açıyoruz.

In [ ]:
!unzip -q -o "{LOCAL_ZIP}" -d "{LOCAL_IMAGES}"

In [ ]:
# Yapıyı görelim
!ls -la "{LOCAL_IMAGES}" | head -10
print('---')
!find "{LOCAL_IMAGES}" -maxdepth 2 -type d | head -20

## 2.5 — Hasta sayısı ve klasör yapısı

PI-CAI yapısı şu şekilde:
```
images/
├── 10000/
│   ├── 10000_1000000_t2w.mha
│   ├── 10000_1000000_adc.mha
│   ├── 10000_1000000_hbv.mha    # high b-value DWI
│   └── 10000_1000000_sag.mha    # sometimes additional sequences
├── 10001/
│   └── ...
```

(Yapı klasör versiyonuna göre değişebilir; bazılarında flat .mha dosyaları olur.)

In [ ]:
# Tüm hastaları listele
import re

patient_dirs = sorted([d for d in LOCAL_IMAGES.iterdir() if d.is_dir()])
print(f'Klasör sayısı: {len(patient_dirs)}')

if len(patient_dirs) == 0:
    # Belki düz dosya yapısındadır
    all_mha = list(LOCAL_IMAGES.rglob('*.mha'))
    print(f'.mha dosyası: {len(all_mha)}')
    # Hasta kimliklerini çıkar (örn: 10000_1000000_t2w.mha → 10000_1000000)
    patient_ids = sorted(set(re.match(r'(\d+_\d+)', p.name).group(1) for p in all_mha if re.match(r'(\d+_\d+)', p.name)))
    print(f'Eşsiz hasta: {len(patient_ids)}')
    print('İlk 5 hasta:', patient_ids[:5])
else:
    print('İlk 5 klasör:', [d.name for d in patient_dirs[:5]])
    sample = patient_dirs[0]
    print(f'\nÖrnek hasta ({sample.name}) içeriği:')
    for f in sample.iterdir():
        print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

## 2.6 — Örnek hastayı görselleştir

Bir hastanın T2W (T2-weighted), ADC (Apparent Diffusion Coefficient), HBV (High B-Value DWI) görüntülerini orta dilimde göster. Bu sunum slaytlarına direkt eklenebilir.

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt

# Bir hasta seç
if patient_dirs:
    sample_dir = patient_dirs[0]
    files = list(sample_dir.glob('*.mha'))
else:
    # flat yapı
    sample_id = patient_ids[0]
    files = list(LOCAL_IMAGES.rglob(f'{sample_id}*.mha'))

print(f'Hasta dosyaları: {[f.name for f in files]}')

# Sequence tipine göre eşleştir
def find_seq(files, suffix):
    for f in files:
        if f.name.lower().endswith(f'_{suffix}.mha'):
            return f
    return None

t2w_path = find_seq(files, 't2w')
adc_path = find_seq(files, 'adc')
hbv_path = find_seq(files, 'hbv')

print(f'T2W: {t2w_path}')
print(f'ADC: {adc_path}')
print(f'HBV: {hbv_path}')

In [ ]:
def load_mha(path):
    if path is None or not path.exists():
        return None
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)  # (Z, Y, X)
    return arr, img

t2w, t2w_img = load_mha(t2w_path) if t2w_path else (None, None)
adc, _       = load_mha(adc_path) if adc_path else (None, None)
hbv, _       = load_mha(hbv_path) if hbv_path else (None, None)

if t2w is not None:
    print(f'T2W shape: {t2w.shape}, dtype: {t2w.dtype}')
    print(f'T2W spacing: {t2w_img.GetSpacing()}')
    print(f'T2W intensity range: [{t2w.min()}, {t2w.max()}]')

In [ ]:
# 3 sequence yan yana göster (orta dilim)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, vol) in zip(axes, [('T2W', t2w), ('ADC', adc), ('HBV (DWI)', hbv)]):
    if vol is None:
        ax.text(0.5, 0.5, f'{name}: yok', ha='center')
        ax.axis('off')
        continue
    mid = vol.shape[0] // 2
    ax.imshow(vol[mid], cmap='gray')
    ax.set_title(f'{name}  (slice {mid}/{vol.shape[0]})')
    ax.axis('off')

plt.suptitle(f'PI-CAI Sample Case — {sample_dir.name if patient_dirs else sample_id}', fontsize=14)
plt.tight_layout()

# Slayt için kaydet
fig_path = OUTPUT_DIR / 'figures' / 'sample_case_sequences.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Kaydedildi: {fig_path}')
plt.show()

## 2.7 — Ground truth (lezyon + anatomi) görselleştir

Aynı hastanın label'ları varsa overlay yap.

In [ ]:
# Aynı hasta için label'ları ara
case_id = sample_dir.name if patient_dirs else sample_id  # e.g. "10000_1000000" veya "10000"

# Lezyon label (human expert)
lesion_human = list((LABELS_DIR / 'csPCa_lesion_delineations' / 'human_expert').rglob(f'{case_id}*.nii.gz'))
# Anatomi label (whole gland, AI)
wg_ai = list((LABELS_DIR / 'anatomical_delineations' / 'whole_gland' / 'AI').rglob(f'{case_id}*.nii.gz'))
# Zonal label (PZ/TZ, AI)
zonal_ai = list((LABELS_DIR / 'anatomical_delineations' / 'zonal_pz_tz' / 'AI').rglob(f'{case_id}*.nii.gz'))

print(f'Lezyon (human) label: {lesion_human}')
print(f'Whole gland (AI):     {wg_ai}')
print(f'Zonal (AI):           {zonal_ai}')

In [ ]:
import nibabel as nib

def load_nii(path_list):
    if not path_list:
        return None
    p = path_list[0]
    return nib.load(str(p)).get_fdata().transpose(2, 1, 0)  # → (Z,Y,X)

lesion_mask = load_nii(lesion_human)
wg_mask     = load_nii(wg_ai)
zonal_mask  = load_nii(zonal_ai)

# T2W üzerine overlay
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
mid = t2w.shape[0] // 2

for ax, (name, mask, cmap) in zip(axes,
    [('Whole Gland (AI)', wg_mask, 'autumn'),
     ('Zonal PZ/TZ (AI)', zonal_mask, 'viridis'),
     ('csPCa Lesion (Human)', lesion_mask, 'Reds')]):
    ax.imshow(t2w[mid], cmap='gray')
    if mask is not None and mid < mask.shape[0]:
        masked = np.ma.masked_where(mask[mid] == 0, mask[mid])
        ax.imshow(masked, cmap=cmap, alpha=0.5)
        ax.set_title(f'{name}\n(slice {mid})')
    else:
        ax.set_title(f'{name}\n(yok)')
    ax.axis('off')

plt.suptitle(f'Ground Truth Overlays — {case_id}', fontsize=14)
plt.tight_layout()
fig_path = OUTPUT_DIR / 'figures' / 'sample_case_gt_overlays.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Kaydedildi: {fig_path}')
plt.show()

## 2.8 — Subset'i Drive'a kalıcılaştır

Fold 0'ın tamamı (~14 GB) Drive'a sığar ama bizim için **20 hasta yeter**. Bunlardan:
- 10 tanesi **csPCa-pozitif** (lezyon var)
- 10 tanesi **csPCa-negatif** (lezyon yok)

Böylece hem pozitif hem negatif örnekler gösterebiliriz.

In [ ]:
import pandas as pd

marksheet = pd.read_csv(LABELS_DIR / 'clinical_information' / 'marksheet.csv')
print(f'Marksheet sütunları: {list(marksheet.columns)}')
marksheet.head()

In [ ]:
# Local imaging klasöründeki hasta listesi
if patient_dirs:
    local_case_ids = [d.name for d in patient_dirs]
else:
    local_case_ids = patient_ids

# Marksheet'te csPCa label sütunu genelde 'case_csPCa' veya 'lesion_GS' veya 'lesion_ISUP' olur
# Önce ne var bakalım:
label_col = None
for cand in ['case_csPCa', 'lesion_GS', 'lesion_ISUP', 'case_ISUP']:
    if cand in marksheet.columns:
        label_col = cand
        break
print(f'Kullanılacak label sütunu: {label_col}')

# patient_id sütununu bul
id_col = None
for cand in ['patient_id', 'case_id', 'study_id']:
    if cand in marksheet.columns:
        id_col = cand
        break
print(f'Kullanılacak id sütunu: {id_col}')

In [ ]:
# csPCa pozitif/negatif ayrımı
if label_col == 'case_csPCa':
    pos_mask = marksheet[label_col] == 'YES'
else:
    pos_mask = marksheet[label_col].fillna(0) > 0

marksheet['patient_str'] = marksheet[id_col].astype(str)

# Local'de bulunan + pozitif/negatif olanlar
local_set = set(str(c).split('_')[0] for c in local_case_ids)

local_pos = marksheet[pos_mask & marksheet['patient_str'].isin(local_set)].head(10)
local_neg = marksheet[~pos_mask & marksheet['patient_str'].isin(local_set)].head(10)

subset_ids = list(local_pos['patient_str']) + list(local_neg['patient_str'])
print(f'Seçilen subset: {len(subset_ids)} hasta')
print(f'  Pozitif: {len(local_pos)}')
print(f'  Negatif: {len(local_neg)}')
print(f'IDs: {subset_ids}')

In [ ]:
import shutil

# Subset'i Drive'a kopyala
DRIVE_SUBSET = IMAGES_DIR / 'subset20'
DRIVE_SUBSET.mkdir(parents=True, exist_ok=True)

copied = 0
for case_root in subset_ids:
    if patient_dirs:
        # klasör yapısı
        matches = [d for d in patient_dirs if d.name.startswith(str(case_root))]
        for src in matches:
            dst = DRIVE_SUBSET / src.name
            if not dst.exists():
                shutil.copytree(src, dst)
                copied += 1
    else:
        # flat yapı
        matches = list(LOCAL_IMAGES.rglob(f'{case_root}*.mha'))
        case_dir = DRIVE_SUBSET / case_root
        case_dir.mkdir(exist_ok=True)
        for src in matches:
            dst = case_dir / src.name
            if not dst.exists():
                shutil.copy(src, dst)
        copied += 1

print(f'{copied} dosya/klasör Drive\'a kopyalandı.')
!du -sh "{DRIVE_SUBSET}"

## 2.9 — Subset envanteri

Drive'a aldığımız 20 hastanın özet tablosunu kaydet — slaytlara koyabilirsin.

In [ ]:
subset_info = pd.concat([local_pos.assign(csPCa='positive'), local_neg.assign(csPCa='negative')])
cols_to_show = [c for c in [id_col, label_col, 'csPCa', 'patient_age', 'psa', 'prostate_volume'] if c in subset_info.columns or c == 'csPCa']
subset_summary = subset_info[cols_to_show].reset_index(drop=True)

out_csv = OUTPUT_DIR / 'metrics' / 'subset20_inventory.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
subset_summary.to_csv(out_csv, index=False)
print(f'Subset envanteri: {out_csv}')
subset_summary

## 2.10 — Temizlik (opsiyonel)

Subset Drive'a aktarıldıktan sonra Colab local'deki zip + unzip artıklarını silebilirsin (session sonunda zaten silinir).

In [ ]:
# UYARI: aşağıdaki yorumu açarsan ~28 GB silinir, geri dönüş yok (yine indirilebilir tabii)
# import shutil
# shutil.rmtree(LOCAL_WORK, ignore_errors=True)
# print('Local çalışma alanı temizlendi.')

---

## ✅ Step 2 Bitti

**Üretildi:**
- `output/figures/sample_case_sequences.png` — 3 sequence görseli (slayt 4)
- `output/figures/sample_case_gt_overlays.png` — ground truth overlay (slayt 5)
- `output/metrics/subset20_inventory.csv` — kullanılacak 20 hastalık subset
- `input/images/subset20/` — Drive'a kalıcılaşmış imaging veri

**Sonraki:** `step3_anatomy_inference.ipynb` — MONAI bundle ile zonal segmentation.